A decorator is a callable that takes another function as an argument (the decorated
function).

In [ ]:
#functions are "first-class citizens." This means you can assign them to variables
def greet(name):
    return f"Hello, {name}!"

say_hi = greet  # Assigning function to a variable
print(say_hi("Alice"))

Hello, Alice!


In [25]:
def outer_function():
    print("I am the outer function.")

    def inner_function():
        print("I am the inner function.")

    inner_function()  # Calling it while still inside outer_function

outer_function()
# If you tried to call inner_function() out here, Python would throw a NameError.

I am the outer function.
I am the inner function.


In [35]:
def create_multiplier(x):
    def multiplier(n):
        print(f'applying {x} multiply by {n}')
        return x * n
    
    return multiplier  # We are returning the FUNCTION object, not calling it!

double = create_multiplier(2) 
# Now 'double' is actually the 'multiplier' function, but it "remembers" x = 2.

print(double(5)) # Output: 10

triple = create_multiplier(3)
print(triple(5))

applying 2 multiply by 5
10
applying 3 multiply by 5
15


In [28]:
# Since functions are objects, we can write a function that takes another function as an input and "wraps" it in new logic.
# A decorator is just a specific version of the code above. Instead of passing an integer like x, we pass a function.

def my_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

def say_hello():
    print("Hello!")

# Manual decoration
decorated_hello = my_decorator(say_hello)
decorated_hello()


Something is happening before the function is called.
Hello!
Something is happening after the function is called.


In [ ]:
# Instead of manually assigning decorated_hello = my_decorator(say_hello), Python provides the @ symbol to do it automatically
@my_decorator
def say_hello():
    print("Hello!")

say_hello()

Something is happening before the function is called.
Hello!
Something is happening after the function is called.


In [44]:
def debug(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args}")
        result = func(*args, **kwargs)
        print(f"func name: {kwargs}")
        return result
    return wrapper

@debug
def add(a, b, name = None):
    return f'result is {a + b}'

add(5,4,name="adding")

Calling add with (5, 4)
func name: {'name': 'adding'}


'result is 9'

In [45]:
def debug(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args}")
        result = func(*args, **kwargs)
        for x , y in kwargs.items():
            print(x,y)
        return result
    return wrapper

@debug
def add(a, b, **c):
    return a + b
add(5,4,name="adding")

Calling add with (5, 4)
name adding


9

In [47]:
# Hijacker

def deco(func):
    def wrapper():
        print('hello')
    return wrapper

@deco
def no():
    print("not hello")
no()


hello


hree essential facts make a good summary of decorators:

• A decorator is a function or another callable.

• A decorator may replace the decorated function with a different one.

• Decorators are executed immediately when a module is loaded

In [51]:
# 1. THE PHONEBOOK (The Registry)
# We create an empty list. This will hold the actual function objects in memory.
registry = [] 

# 2. THE DECORATOR
# Notice: There is no 'def wrapper():' inside here!
def register(func):
    print(f'Import Time: Adding {func.__name__} to the registry...')
    
    # We append the original, untouched function to our list.
    registry.append(func) 
    
    # CRITICAL STEP: We return the EXACT SAME function back.
    # We didn't wrap it. We didn't change it. 
    return func 

# --- IMPORT TIME HAPPENS HERE ---
# As soon as Python reads the line below, it immediately executes `register(f1)`
@register
def f1():
    print('Runtime: f1 is doing its job.')

# Python reads this and immediately executes `register(f2)`
@register
def f2():
    print('Runtime: f2 is doing its job.')

def f3():
    print('Runtime: f3 is doing its job.')
# (f3 is ignored by the decorator because it doesn't have the @ symbol)

# --- RUNTIME HAPPENS HERE ---
def main():
    print("--- Starting Runtime ---")
    print("What is in our registry?", registry)
    
    # Now we actually execute the functions
    f1()
    f2()

if __name__ == '__main__':
    main()

Import Time: Adding f1 to the registry...
Import Time: Adding f2 to the registry...
--- Starting Runtime ---
What is in our registry? [<function f1 at 0x0000026C096876A0>, <function f2 at 0x0000026C09687060>]
Runtime: f1 is doing its job.
Runtime: f2 is doing its job.


decorator can do three things:

Enhance: Run code before/after the original function (most common).

Replace: Run completely different code and ignore the original function.

Filter: Only run the original function if a condition is met (like checking a password or a timer).

When you run a Python script, Python reads it from top to bottom. When it sees a def, it creates a function object in memory.

But when it sees a @decorator, it executes that decorator immediately.

In [52]:
from fastapi import FastAPI

# 1. This creates the "Phonebook" (The Registry)
# Deep inside the FastAPI code, 'app' has an empty dictionary to store routes.
app = FastAPI() 

# --- IMPORT TIME ---

# 2. The Decorator Factory
# @app.get is a decorator that takes a URL string as an argument.
@app.get("/home")
def show_homepage():
    return {"message": "Welcome to my homepage!"}

@app.get("/profile")
def show_user_profile():
    return {"user": "Alice", "status": "Online"}

# --- RUNTIME ---
# (Imagine a user opens their browser and visits your website)

1. At Import Time (When you start the server):
Before any user ever visits your website, Python reads your file from top to bottom.

It sees @app.get("/home").

It immediately takes the show_homepage function and stores it in FastAPI's internal registry, linked to the string "/home".

It sees @app.get("/profile").

It stores the show_user_profile function, linked to the string "/profile".

FastAPI's internal dictionary now looks something like this:
{"/home": show_homepage, "/profile": show_user_profile}

Notice that FastAPI doesn't change your functions. It just records them. It's a wrapper-less decorator, just like Example 9-2!

2. At Runtime (When a user visits the website):

A user clicks a link to yourwebsite.com/profile.

The web server receives the request for the "/profile" path.

FastAPI looks at its registry (the dictionary it built at import time).

It says, "Ah, the user wants /profile. According to my registry, I need to execute the show_user_profile function!"

It runs the function and sends the JSON {"user": "Alice", "status": "Online"} back to the user's browser.

most decorators DO use an inner wrapper function. To understand how inner wrappers work, you have to understand Closures. And to understand Closures, we first need to look at Variable Scopes

variable scope rules:

In [2]:
b = 6 # Global variable
def f1(a):
    print(a)
    print(b) # Python looks locally, finds nothing, so it looks globally and finds 6.
f1(1)

1
6


In [4]:
b = 6
def f2(a):
    print(a)
    print(b) # BOOM! UnboundLocalError
    b = 9    # This assignment changes everything
f2(1)

1


UnboundLocalError: cannot access local variable 'b' where it is not associated with a value

Why does it crash?
Because Python does not read your code line-by-line like a blind robot. Before it runs the function, it compiles the whole body. It scans the function and sees b = 9. It says, "Ah! The programmer is assigning a value to b inside this function. Therefore, b is a LOCAL variable for this entire function."

When it actually runs print(b), it looks at its local memory. It sees that local b exists, but you haven't assigned the 9 to it yet. Because it considers b strictly local, it refuses to look outside at the global b = 6. It throws an error: local variable 'b' referenced before assignment.

In [5]:
#the fix
b = 6
def f3(a):
    global b # "Hey Python, don't make a local b. Use the one outside."
    print(a)
    print(b)
    b = 9    # This now updates the global b to 9
f3(1)

1
6


To prove that Python makes up its mind about variable scope before the code runs, the author uses the dis (disassembler) module.

Python code isn't executed directly by your CPU. It is first compiled into intermediate instructions called bytecode (which the Python Virtual Machine then runs).

By looking at the bytecode, you can read Python's mind.

A lot of people think "closure" is just a fancy word for an "anonymous function" (like a lambda).

 No. A closure has nothing to do with whether a function has a name or not.

Definition: A closure is simply an inner function that "remembers" variables from its outer function, even after the outer function has finished running and died.

In [29]:
class Averager():
    def __init__(self):
        self.series = [] # <--- Here is the memory!
        
    def __call__(self, new_value):
        self.series.append(new_value)
        total = sum(self.series)
        return total / len(self.series)

ser = Averager()
ser(1)
ser(2)
ser(3)

2.0

In [30]:
def make_averager():
    series = [] # <--- This is a local variable inside the outer function

    def averager(new_value):
        series.append(new_value) # <--- The inner function uses it!
        total = sum(series)
        return total / len(series)

    return averager # Hand back the inner function

he Magic of Closures:
When Python sees that the inner function needs the series variable, it creates an invisible "backpack" and attaches it to the inner function. It puts the series list inside this backpack. So, even though the outer function dies, the inner function walks away carrying the series list in its closure backpack!



The "Backpack" (Free Variables)

In [34]:
avg = make_averager()
avg(10)
avg(20)

15.0

In [38]:
def new_averager():
    total = 0
    count = 0
    def avv(new):
        count += 1
        total += new
        return total / count
    return avv

aver = new_averager()
aver(5)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

If you are using = to completely overwrite an immutable variable (like numbers, strings, or booleans), you must use nonlocal, or Python will accidentally create a broken local variable instead.

In [11]:
# Level 1: The Box
def create_counter():
    count = 0  # This is our memory box
    
    # Level 2: The Action
    def click():
        nonlocal count  # "Python, let me change the box above me!"
        count = count + 1
        print(f"Clicked {count} times")
        
    return click  # Hand back the action

# Let's test it:
my_button = create_counter()

my_button() # Output: Clicked 1 times
my_button() # Output: Clicked 2 times

Clicked 1 times
Clicked 2 times


In [12]:
my_button()

Clicked 3 times


Level 1 (The Factory): Takes your argument (e.g., "Abdelrahman").

Level 2 (The Decorator): Takes the function.

Level 3 (The Wrapper): Runs the code.

In [ ]:
# LEVEL 1: Takes your custom argument
def custom_greeting(name):
    
    # LEVEL 2: Takes the function (this is the actual decorator)
    def decorator(func):
        
        # LEVEL 3: The wrapper that runs the code
        def wrapper():
            print(f"Hello {name}!")
            return func()
            
        return wrapper
    return decorator

# When Python reads this, it runs custom_greeting("Abdelrahman") FIRST.
# That creates Level 2, which then wraps 'do_work'.
@custom_greeting("Abdelrahman")
def do_work():
    print("Doing work now.")

do_work()
# Outp

Hello Abdelrahman!
Doing work now.


golbal cant work here we need something for the nested function and it is nonlocal

In [20]:
import time
from functools import wraps

# LEVEL 1: The Factory (Takes the configuration)
def rate_limit(seconds):
    # This variable lives in the "closure" memory.
    # It remembers the state between function calls.
    last_called = 0.0 

    # LEVEL 2: The Decorator (Takes the function)
    def decorator(func):
        
        @wraps(func) # Preserves the original function's name
        # LEVEL 3: The Wrapper (Executes the logic)
        def wrapper(*args, **kwargs):
            
            # Remember the Variable Scope lesson? 
            # We use 'nonlocal' to tell Python: "Don't create a new local 
            # variable, use the 'last_called' from Level 1."
            nonlocal last_called 
            
            current_time = time.time()
            time_passed = current_time - last_called
            
            # The Logic
            if time_passed >= seconds:
                # Enough time has passed. Update the timestamp and run!
                last_called = current_time
                return func(*args, **kwargs)
            else:
                # Not enough time has passed. Block it!
                time_left = seconds - time_passed
                print(f"Rate limit exceeded! Please wait {time_left:.1f} more seconds.")
                
        return wrapper # Level 2 hands back Level 3
    return decorator   # Level 1 hands back Level 2


# --- Let's test it out! ---

@rate_limit(5)
def generate_gat_question(topic):
    print(f"Generating new interactive {topic} question for the Qudrat exam...")
    # Imagine AI generation logic happens here
    return "Question generated successfully!"

# First click: Works immediately (time_passed is huge because last_called was 0)
generate_gat_question("Quantitative") 




Generating new interactive Quantitative question for the Qudrat exam...


'Question generated successfully!'

When you add parentheses and an argument to a decorator, it is no longer a decorator. It is a decorator factory

In [21]:
generate_gat_question("Verbal") 


Rate limit exceeded! Please wait 3.5 more seconds.


The "Look Outward" Search (LEGB Rule): If x isn't assigned locally, Python starts looking outward in expanding circles:

Local: Inside the current function? (No).

Enclosing: Inside the backpack (Closure) from a parent function?

Global: At the very top of the file?

Built-in: Is it a built-in Python word (like print or len)?

## Memoization and @cache

In [50]:
def fab(n):
    return n if n <2 else fab(n-1) + fab(n-2)

fab(10)

55

Python gives you a decorator called @cache (found in the functools module) that does exactly this.

When you put @cache above your function:

Python creates a hidden, empty dictionary in the decorator's closure backpack.

When you call fib(4), it checks the dictionary. It's not there.

It does the hard math, gets the answer 3, and saves it in the dictionary: {4: 3}.

The next time the computer needs fib(4), the decorator intercepts it, says "I already know this!", and instantly returns 3 without running the original function at all.

By just adding that one @cache line, the fib(30) calculation drops from 12 seconds to 0.00017 seconds. It is pure magic.

In [ ]:
# This is read from the bottom up like unrapping a present
import functools
from clockdeco import clock

@functools.cache
@clock
def fibonacci(n):
    return n if n <2 else fab(n-1) + fab(n-2)
fibonacci(10)

## important

@cache is dangerous for long-running programs (like a FastAPI web server) because it never forgets. If a million users type a million different things into your app, @cache will save all one million answers in its hidden dictionary. Your server will run out of RAM and crash.

The Fix: @lru_cache
LRU stands for Least Recently Used. It is the exact same thing as @cache, but it has a built-in memory limit using the maxsize parameter.

In [56]:
#Not efficint code

def display_data(data):
    if isinstance(data, str):
        print(f"String: {data}")
    elif isinstance(data, int):
        print(f"Number: {bin(data)}")
    elif isinstance(data, list):
        for item in data:
            print(f" - {item}")
    else:
        print(f"Unknown data: {data}")

### The Solution (@singledispatch)
In languages like Java or C++, you can write multiple functions with the exact same name as long as they take different types of arguments (this is called "Method Overloading").

Python doesn't allow that. But @singledispatch gives us something even better. It acts like a Traffic Cop.

Instead of one giant function, you write a bunch of small, separate functions. The decorator automatically routes the data to the correct function based on the type of the first argument.

In [57]:
# DEfAULT

from functools import singledispatch

@singledispatch
def display_data(data):
    # This is the "catch-all" or default behavior
    print(f"Unknown type: {data}")

In [58]:
# 1. The String Specialist
@display_data.register
def _(text: str):  # Python sees ': str' and routes strings here
    print(f"String: {text}")

# 2. The Integer Specialist
@display_data.register
def _(number: int): # Python sees ': int' and routes integers here
    print(f"Number: {bin(number)}")

# 3. The List Specialist
@display_data.register
def _(items: list): # Python sees ': list' and routes lists here
    for item in items:
        print(f" - {item}")

If a user or a developer wanna modify just:


from your_module import display_data

from math_tools import Matrix

@display_data.register

def _(m: Matrix):

    print("Printing a beautiful 3D matrix...")

In [3]:
import time
from functools import wraps

class RateLimitTracker:
    # 1. THE FACTORY
    # Sets up the rules and the memory when @RateLimitTracker(seconds=5) is read.
    def __init__(self, seconds):
        self.seconds = seconds
        self.last_called = 0.0  # Our memory box is safely stored in 'self'
        
    # 2. THE DECORATOR
    # Python passes the function into this __call__ magic method.
    def __call__(self, func):
        
        @wraps(func)
        # 3. THE WRAPPER
        def wrapper(*args, **kwargs):
            current_time = time.time()
            time_passed = current_time - self.last_called
            
            # THE LOGIC
            if time_passed >= self.seconds:
                # Update the object's memory and run the AWS function
                self.last_called = current_time
                return func(*args, **kwargs)
            else:
                # Block the execution
                time_left = self.seconds - time_passed
                print(f"Rate limit hit! To protect the AWS bill, wait {time_left:.1f}s.")
                # We return None implicitly here, or you could raise an Exception
                
        return wrapper


# --- Test it on the RAG pipeline ---

@RateLimitTracker(seconds=5)
def query_rag_pipeline(user_prompt):
    print(f"Fetching context and generating answer for: '{user_prompt}'")
    return "LLM Response"

print("--- First Call ---")
query_rag_pipeline("What is machine learning?") 




--- First Call ---
Fetching context and generating answer for: 'What is machine learning?'


'LLM Response'

In [4]:
print("\n--- Second Call (Immediately) ---")
query_rag_pipeline("Tell me more.") 


--- Second Call (Immediately) ---
Rate limit hit! To protect the AWS bill, wait 3.1s.


No more nonlocal! Remember how we had to use nonlocal last_called in our Rate Limiter to stop it from crashing? In a class, you don't need closures or nonlocal. You just store your variables in self (like self.last_called = 0). It is much safer and easier to read.

Flatter Code. Instead of three functions indented so far to the right that they fall off your screen, you only have two levels of indentation inside the class.

State Tracking. If your decorator needs a complicated memory (like tracking the average time of the last 100 function calls), managing that state inside a Class using self is infinitely easier than managing it inside nested function closures.